# 02 — Data Quality Check

**Purpose:** Assess dataset readiness before feature engineering.

**Checks:**
1. Null rates per column
2. Class balance (`at_risk` label preview)
3. Learner group distribution
4. Row counts and unique student/task coverage
5. Batch coverage

**Input:** `df_session` and `df_attempt` from `01_load_dataset.ipynb`

**Output:** Quality report printed in-notebook. Proceed to `03_feature_engineering.ipynb` only if all checks pass.

## Imports and load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Update to match snapshot filenames from 01_load_dataset.ipynb ─────────────
SNAPSHOT_DATE = "YYYY-MM-DD"
BATCH_CODE    = "BATCH_XXX"

SESSION_FILE = f"notebooks/data/raw/session_{SNAPSHOT_DATE}_{BATCH_CODE}.csv"
ATTEMPT_FILE = f"notebooks/data/raw/attempt_{SNAPSHOT_DATE}_{BATCH_CODE}.csv"

df_session = pd.read_csv(SESSION_FILE)
df_attempt = pd.read_csv(ATTEMPT_FILE)

print(f"Session rows : {len(df_session):,}")
print(f"Attempt rows : {len(df_attempt):,}")

## 1. Null rates — session level

In [ ]:
null_session = (
    df_session.isnull().sum()
    .to_frame("null_count")
    .assign(null_pct=lambda d: d["null_count"] / len(df_session) * 100)
    .query("null_count > 0")
    .sort_values("null_pct", ascending=False)
)

if null_session.empty:
    print("[OK] No nulls in session data")
else:
    print(f"[WARN] {len(null_session)} columns have nulls:")
    display(null_session)

## 2. Null rates — attempt level

In [ ]:
null_attempt = (
    df_attempt.isnull().sum()
    .to_frame("null_count")
    .assign(null_pct=lambda d: d["null_count"] / len(df_attempt) * 100)
    .query("null_count > 0")
    .sort_values("null_pct", ascending=False)
)

if null_attempt.empty:
    print("[OK] No nulls in attempt data")
else:
    print(f"[WARN] {len(null_attempt)} columns have nulls:")
    display(null_attempt)

## 3. Class balance — at_risk label preview

Label definition: `at_risk = 1` when `COALESCE(review_score, auto_score) < max_score * 0.6`

Rows with no submission (both scores null) are also labeled `at_risk = 1`.

In [ ]:
PASS_THRESHOLD_RATIO = 0.6

df_session["effective_score"] = df_session["review_score"].combine_first(df_session["auto_score"])
df_session["pass_threshold"]  = df_session["max_score"] * PASS_THRESHOLD_RATIO
df_session["at_risk"] = (
    df_session["effective_score"].isna() |
    (df_session["effective_score"] < df_session["pass_threshold"])
).astype(int)

balance = df_session["at_risk"].value_counts().rename(index={0: "not_at_risk", 1: "at_risk"})
balance_pct = balance / len(df_session) * 100

print("Class balance (at_risk label):")
print(pd.DataFrame({"count": balance, "pct": balance_pct.round(1)}))

ratio = balance.get("not_at_risk", 0) / max(balance.get("at_risk", 1), 1)
print(f"\nImbalance ratio (not_at_risk : at_risk) = {ratio:.2f} : 1")
if ratio > 3:
    print("[WARN] Imbalance ratio > 3 — class_weight='balanced' will be applied in training")
else:
    print("[OK] Imbalance ratio within acceptable range")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
balance.plot(kind="bar", ax=ax, color=["steelblue", "tomato"], edgecolor="white")
ax.set_title("Class balance — at_risk label")
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.bar_label(ax.containers[0], fmt="%d")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("notebooks/reports/class_balance.png", dpi=150)
plt.show()

## 4. Learner group distribution

In [ ]:
if "learner_group" not in df_session.columns:
    print("[WARN] 'learner_group' column not found — check export view")
else:
    lg_dist = df_session["learner_group"].value_counts().sort_index()
    lg_pct  = lg_dist / len(df_session) * 100
    print("Learner group distribution:")
    print(pd.DataFrame({"count": lg_dist, "pct": lg_pct.round(1)}))

    fig, ax = plt.subplots(figsize=(5, 3))
    lg_dist.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title("Learner group distribution")
    ax.set_xlabel("Learner group")
    ax.set_ylabel("Count")
    ax.bar_label(ax.containers[0], fmt="%d")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig("notebooks/reports/learner_group_dist.png", dpi=150)
    plt.show()

## 5. Row counts and coverage

In [ ]:
n_students = df_session["academy_member_id"].nunique()
n_tasks    = df_session["task_id"].nunique()
n_pairs    = df_session[["academy_member_id", "task_id"]].drop_duplicates().shape[0]
max_pairs  = n_students * n_tasks

print(f"Unique students          : {n_students:,}")
print(f"Unique tasks             : {n_tasks:,}")
print(f"Student × task pairs     : {n_pairs:,} / {max_pairs:,} ({n_pairs/max_pairs*100:.1f}% coverage)")

if n_students < 30:
    print(f"[WARN] Only {n_students} unique students — GroupShuffleSplit may produce unstable splits")

## 6. Batch coverage

In [ ]:
if "batch_code" not in df_session.columns:
    print("[WARN] 'batch_code' column not found")
else:
    batch_dist = df_session["batch_code"].value_counts()
    print(f"Batches found: {len(batch_dist)}")
    print(batch_dist.to_frame("session_count"))

    if len(batch_dist) >= 2:
        print("\n[INFO] Multiple batches detected — temporal/batch split is available as secondary split in 03_feature_engineering.ipynb")
    else:
        print("\n[INFO] Single batch — primary GroupShuffleSplit only")

## Quality gate summary

In [ ]:
print("=" * 50)
print("Data quality gate")
print("=" * 50)
print(f"  Null cols (session) : {len(null_session)}")
print(f"  Null cols (attempt) : {len(null_attempt)}")
print(f"  Imbalance ratio     : {ratio:.2f} : 1")
print(f"  Unique students     : {n_students}")
print(f"  Pair coverage       : {n_pairs/max_pairs*100:.1f}%")
print("=" * 50)

blockers = []
if n_students < 10:
    blockers.append("Too few students for reliable split")
if n_pairs == 0:
    blockers.append("No student × task pairs found")

if blockers:
    print("[BLOCKED] Cannot proceed to feature engineering:")
    for b in blockers:
        print(f"  - {b}")
else:
    print("[OK] Proceed to 03_feature_engineering.ipynb")